# Customer Churn Prediction
**Internship Project | Data Science | Classification**

## Objective
Identify customers who may leave a telecommunications service and understand which customer characteristics are associated with churn.

## Dataset
IBM Telco Customer Churn sample dataset (7,043 customer records). The notebook downloads the public CSV from IBM's GitHub repository when run in Google Colab or another internet-enabled Python environment.

**Target:** `Churn` (`Yes` = customer left; `No` = customer stayed).

> Educational portfolio project only. This is a historical sample and must not be treated as a production decision system.


In [ ]:
# Install dependencies if needed (uncomment in a fresh environment)
# %pip install pandas numpy matplotlib seaborn scikit-learn

import warnings
warnings.filterwarnings("ignore")

import os
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, ConfusionMatrixDisplay,
    RocCurveDisplay
)

RANDOM_STATE = 42
os.makedirs("outputs", exist_ok=True)
sns.set_theme(style="whitegrid")


## 1. Load the data

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
DATA_PATH = "Telco-Customer-Churn.csv"

if not os.path.exists(DATA_PATH):
    try:
        urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    except Exception as exc:
        raise RuntimeError(
            "Could not download the dataset. Download Telco-Customer-Churn.csv "
            "from the IBM repository and upload it beside this notebook."
        ) from exc

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())
display(df.info())


## 2. Data quality and preparation

In [ ]:
print("Duplicate customer IDs:", df["customerID"].duplicated().sum())
print("\\nMissing values by column:")
display(df.isna().sum().sort_values(ascending=False).head(10))

# TotalCharges is stored as text in the source; blank values correspond to new customers.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0}).astype(int)

# Remove the identifier because it is not a meaningful predictive feature.
model_df = df.drop(columns=["customerID"]).copy()
print("Churn rate: {:.1%}".format(model_df["Churn"].mean()))
display(model_df.describe(include="all").T.head(20))


## 3. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.countplot(data=df, x="Churn", ax=axes[0])
axes[0].set_title("Customer Churn Counts")
axes[0].set_xlabel("Churn")
sns.histplot(data=df, x="tenure", hue="Churn", bins=24, kde=True, element="step", ax=axes[1])
axes[1].set_title("Tenure Distribution by Churn")
plt.tight_layout()
plt.savefig("outputs/churn_eda.png", dpi=160, bbox_inches="tight")
plt.show()

contract_rates = (df.groupby("Contract")["Churn"]
                  .apply(lambda s: (s == "Yes").mean() if s.dtype == "object" else s.mean())
                  .sort_values(ascending=False))
# Calculate churn rate directly from the original labels for clear interpretation.
contract_rates = df.assign(churn_flag=(df["Churn"] == "Yes").astype(int)).groupby("Contract")["churn_flag"].mean().sort_values(ascending=False)
plt.figure(figsize=(8, 4))
sns.barplot(x=contract_rates.index, y=contract_rates.values)
plt.ylabel("Churn rate")
plt.xlabel("Contract type")
plt.title("Churn Rate by Contract Type")
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig("outputs/churn_by_contract.png", dpi=160, bbox_inches="tight")
plt.show()

service_rates = df.assign(churn_flag=(df["Churn"] == "Yes").astype(int)).groupby("InternetService")["churn_flag"].mean().sort_values(ascending=False)
display((service_rates * 100).round(1).rename("Churn rate (%)").to_frame())


## 4. Split data and build preprocessing pipelines

In [ ]:
X = model_df.drop(columns=["Churn"])
y = model_df["Churn"]

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Training rows:", len(X_train), "| Test rows:", len(X_test))


## 5. Train and evaluate classification models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, min_samples_leaf=2, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1
    )
}

results = []
fitted_models = {}
for name, estimator in models.items():
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", estimator)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, proba)
    })
    fitted_models[name] = pipe

metrics = pd.DataFrame(results).sort_values("ROC-AUC", ascending=False)
display(metrics.round(3))
metrics.to_csv("outputs/model_metrics.csv", index=False)
best_name = metrics.iloc[0]["Model"]
best_model = fitted_models[best_name]
best_pred = best_model.predict(X_test)
best_proba = best_model.predict_proba(X_test)[:, 1]
print("Best model by ROC-AUC:", best_name)
print(classification_report(y_test, best_pred, target_names=["Stayed", "Churned"], zero_division=0))


## 6. Confusion matrix and ROC curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, best_pred, display_labels=["Stayed", "Churned"], cmap="Blues", ax=axes[0]
)
axes[0].set_title(f"Confusion Matrix — {best_name}")
RocCurveDisplay.from_predictions(y_test, best_proba, ax=axes[1], name=best_name)
axes[1].plot([0, 1], [0, 1], linestyle="--", color="gray")
axes[1].set_title("ROC Curve")
plt.tight_layout()
plt.savefig("outputs/model_evaluation.png", dpi=160, bbox_inches="tight")
plt.show()


## 7. Feature importance / interpretation

In [ ]:
# Random Forest feature importance is useful for a simple, interpretable ranking.
rf_pipe = fitted_models["Random Forest"]
feature_names = rf_pipe.named_steps["preprocessor"].get_feature_names_out()
importances = rf_pipe.named_steps["model"].feature_importances_
importance_df = pd.DataFrame({"Feature": feature_names, "Importance": importances})
importance_df = importance_df.sort_values("Importance", ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df, x="Importance", y="Feature")
plt.title("Top 15 Random Forest Feature Importances")
plt.tight_layout()
plt.savefig("outputs/feature_importance.png", dpi=160, bbox_inches="tight")
plt.show()
display(importance_df)
importance_df.to_csv("outputs/feature_importance.csv", index=False)


## 8. Export a sample churn-risk list

In [ ]:
risk_output = X_test.copy()
risk_output["Actual_Churn"] = y_test.values
risk_output["Predicted_Churn_Probability"] = best_proba
risk_output["Predicted_Churn"] = (best_proba >= 0.5).astype(int)
risk_output = risk_output.sort_values("Predicted_Churn_Probability", ascending=False)
risk_output.head(100).to_csv("outputs/top_100_churn_risk_customers.csv", index=False)
display(risk_output.head(10)[["tenure", "Contract", "MonthlyCharges", "Predicted_Churn_Probability", "Predicted_Churn", "Actual_Churn"]])


## 9. Business insights and conclusion

Use the charts and metrics above to write conclusions from the actual run. Typical questions to answer:

- Which contract type has the highest observed churn rate?
- Does churn appear more common among newer customers?
- Which model has the best ROC-AUC, and what are its recall and precision?
- If recall is important, how many churners does the model identify and how many does it miss?
- What retention actions could be tested (e.g., onboarding support or contract options)?

**Important limitations:** This dataset is a sample snapshot, not a time-series forecast. Feature importance shows association, not causation. Do not contact or make consequential decisions about real customers solely from this educational model.

### Output files
Charts and CSV reports are saved in the `outputs/` folder.
